# 02 — Vector Search Index

**Purpose:** Embed `docs.policy_documents` into a Mosaic AI Vector Search Delta Sync
Index so policy profiles can be retrieved by meaning, not just exact match — the retrieval
half of the RAG chain.

### What This Notebook Does
1. Creates the single AI Search endpoint Free Edition allows (reused if it already exists).
2. Creates a Delta Sync Index over `policy_text`, embedded with the Databricks-hosted
   `databricks-gte-large-en` model — Free Edition only supports Delta Sync (Direct Vector
   Access indexes are not available).
3. Runs a sample similarity search to validate retrieval quality.

### Source & Target
| Direction | Table / Resource |
|-----------|-------------------|
| Source | `insurance_rag_agent.docs.policy_documents` |
| Target | `insurance_rag_agent.docs.policy_documents_index` |

> **Prerequisites:** Run `src/01_prepare_policy_documents` first.

---

In [ ]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG                   = 'insurance_rag_agent'
SOURCE_TABLE               = f'{CATALOG}.docs.policy_documents'
VS_ENDPOINT_NAME           = 'insurance_rag_agent_vs_endpoint'
INDEX_NAME                 = f'{CATALOG}.docs.policy_documents_index'
EMBEDDING_MODEL_ENDPOINT   = 'databricks-gte-large-en'
SAMPLE_QUERY               = 'smokers in the southeast region with high insurance charges'

print(f'Source table: {SOURCE_TABLE}')
print(f'Vector Search endpoint: {VS_ENDPOINT_NAME}')
print(f'Index: {INDEX_NAME}')

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

---

In [0]:
existing_endpoints = [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]

print(f'Existing AI Search endpoints: {existing_endpoints}')

---

In [0]:
# Free Edition allows exactly one AI Search endpoint — reuse it if it already exists
if VS_ENDPOINT_NAME not in existing_endpoints:
    vsc.create_endpoint(name=VS_ENDPOINT_NAME, endpoint_type='STANDARD')

try:
    index = vsc.get_index(endpoint_name=VS_ENDPOINT_NAME, index_name=INDEX_NAME)
    index.sync()
except Exception:
    index = vsc.create_delta_sync_index(
        endpoint_name                 = VS_ENDPOINT_NAME,
        source_table_name             = SOURCE_TABLE,
        index_name                    = INDEX_NAME,
        pipeline_type                 = 'TRIGGERED',
        primary_key                   = 'policy_id',
        embedding_source_column       = 'policy_text',
        embedding_model_endpoint_name = EMBEDDING_MODEL_ENDPOINT,
    )

print(f'Index ready: {INDEX_NAME}')

---

In [0]:
results = index.similarity_search(
    query_text = SAMPLE_QUERY,
    columns    = ['policy_id', 'policy_text', 'region', 'smoker', 'charges_tier'],
    num_results = 5,
)

for row in results['result']['data_array']:
    print(row)